# 1、TodoListMiddleWare中间件

In [1]:
import os

from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_experimental.graph_transformers.llm import system_prompt

load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [4]:
from pathlib import Path
import subprocess
from langchain.tools import tool

# 建议使用绝对路径，避免相对路径带来的混淆
WORKSPACE = Path("../todo_workspace")
# 如果希望从环境变量读取，可以：


@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，必须指向目录，默认为 '.' 表示工作区根路径，
              不能访问工作区外的目录。
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    # 路径穿越检查
    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"

    if not target.exists():
        return f"错误：目录不存在: {path}"
    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)


@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件相对路径。
    """
    file_path = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(file_path).startswith(str(workspace_root)):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    if not file_path.is_file():
        return f"错误：路径不是文件: {path}"

    try:
        return file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return "错误：文件可能是二进制文件，无法以文本方式读取。"


@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件相对路径。
        content: 写入文件的内容。
    """
    file_path = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(file_path).startswith(str(workspace_root)):
        return "错误：只允许写入工作区内的文件。"

    # 确保父目录存在
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为：
    returncode=0|1
    STDOUT:
    ...
    STDERR:
    ...
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except subprocess.TimeoutExpired:
        return "错误：测试运行超时（20秒）"
    except FileNotFoundError:
        return "错误：未找到 pytest 命令，请确认已安装（pip install pytest）"
    except Exception as e:
        return f"运行测试失败: {e}"



In [6]:
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich import print as rprint
agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file,run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt="你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
"然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"

)

response = agent.invoke({
    "messages":[HumanMessage("请测试并修复工作区下的my_add.py文件中的代码")]
})
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下的my_add.py文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='ec30a133-0edc-47f2-ae82-f5fc4d1042c9'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 42,
                    'prompt_tokens': 1762,
                    'total_tokens': 1804,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 1762
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'a4adb254-d8e9-4c70-a7b5-e954a7117fee',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0849e-0b6b-70c1-9269-cf25fa97f8e1-0',
            tool_calls=[
                {
                    'name': 'list_files',
                    'args': {'path': '.'},
                    'id': 'call_00_Zbj3t7aQu0gOwIEOIesV7888',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1762,
                'output_tokens': 42,
                'total_tokens': 1804,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='[DIR] .pytest_cache\n[DIR] __pycache__\n[FILE] my_add.py\n[FILE] test_my_add.py',
            name='list_files',
            id='e432bd01-0307-454b-9957-75d26886f92c',
            tool_call_id='call_00_Zbj3t7aQu0gOwIEOIesV7888'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 79,
                    'prompt_tokens': 1845,
                    'total_tokens': 1924,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1792},
                    'prompt_cache_hit_tokens': 1792,
                    'prompt_cache_miss_tokens': 53
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'c1b8c619-97de-4cfe-aabd-05b59d19f993',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0849e-0e85-7210-b38e-514482c5aa1a-0',
            tool_calls=[
                {
                    'name': 'read_file',
                    'args': {'path': 'my_add.py'},
                    'id': 'call_00_di0FMWIFb7MshFwL2h8Z4675',
                    'type': 'tool_call'
                },
                {
                    'name': 'read_file',
                    'args': {'path': 'test_my_add.py'},
                    'id': 'call_01_M8QCs5vpNYuhg4tD0twT8414',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1845,
                'output_tokens': 79,
                'total_tokens': 1924,
                'input_token_details': {'cache_read': 1792},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='def add(a: int, b: int) -> int:\n    """返回两个整数的和"""\n    return a - b',
            name='read_file',
            id='f4431926-2824-4f9b-86c9-0b60245280b5',
            tool_call